# Synthetic (avgdeg=3) HMRW Experiments

Runs resumable sweeps on synthetic Hy-MMSBM data:

`data/synthetic/hymmsbm/hy_mmsbm_avgdeg_3.pickle`

Sweeps: `p`, `q`, `r`, `num_walks`, `walk_length`, `dim`.


In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FormatStrFormatter

from hypergraphx import Hypergraph
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.utils import hmrw_embeddings


In [2]:
def evaluate_embeddings(X, y, seed=42, n_splits=5):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    clf = LogisticRegression(max_iter=3000)
    scores = cross_validate(
        clf,
        X,
        y,
        cv=cv,
        scoring={"micro_f1": "f1_micro", "macro_f1": "f1_macro"},
        n_jobs=-1,
        return_train_score=False,
    )
    return float(np.mean(scores["test_micro_f1"])), float(np.mean(scores["test_macro_f1"]))


In [3]:
data_path = Path("data") / "synthetic" / "hymmsbm" / "hy_mmsbm_avgdeg_3.pickle"
with open(data_path, "rb") as f:
    bundle = pickle.load(f)

edge_list = bundle["edge_list"]
y = np.asarray(bundle["labels"], dtype=int)
H = Hypergraph(edge_list)

print(f"Loaded: {data_path}")
print(f"nodes={len(H.get_nodes())}, edges={len(H.get_edges())}, labels={len(y)}")


Loaded: data/synthetic/hymmsbm/hy_mmsbm_avgdeg_3.pickle
nodes=825, edges=407, labels=825


In [4]:
results_dir = Path("results") / "synthetic" / "avgdeg3"
results_dir.mkdir(parents=True, exist_ok=True)

base_params = dict(
    p=1.0,
    q=1.0,
    r=1.0,
    num_walks=10,
    walk_length=40,
    window=10,
    dim=128,
    workers=1,
)

n_repeats = 5
base_seed = 42


In [5]:
def run_resumable_sweep(varying_param, values, fixed_params, out_csv):
    out_csv = Path(out_csv)
    if out_csv.exists():
        df = pd.read_csv(out_csv)
    else:
        df = pd.DataFrame(columns=[
            varying_param,
            "micro_f1_mean", "micro_f1_std",
            "macro_f1_mean", "macro_f1_std",
            "n_repeats",
        ])

    done = set(df[varying_param].tolist()) if varying_param in df.columns else set()

    for val in values:
        if val in done:
            print(f"skip {varying_param}={val}")
            continue

        print(f"run {varying_param}={val}")
        micro_runs, macro_runs = [], []
        for rep in range(n_repeats):
            seed = base_seed + rep
            params = dict(fixed_params)
            params[varying_param] = val
            X = hmrw_embeddings(H, seed=seed, **params)
            micro, macro = evaluate_embeddings(X, y, seed=seed)
            micro_runs.append(micro)
            macro_runs.append(macro)

        row = {
            varying_param: val,
            "micro_f1_mean": float(np.mean(micro_runs)),
            "micro_f1_std": float(np.std(micro_runs)),
            "macro_f1_mean": float(np.mean(macro_runs)),
            "macro_f1_std": float(np.std(macro_runs)),
            "n_repeats": n_repeats,
        }
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        df.to_csv(out_csv, index=False)

    return pd.read_csv(out_csv).sort_values(varying_param)


## Sweep q

In [6]:
q_values = [0.25, 0.5, 1.0, 2.0, 4.0]
q_csv = results_dir / "hmrw_q_sweep.csv"
results_q = run_resumable_sweep(
    varying_param="q",
    values=q_values,
    fixed_params={k: v for k, v in base_params.items() if k != "q"},
    out_csv=q_csv,
)
results_q

run q=0.25


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
/var/folders/8z/x99j21nn30vcxmd912tgx33r0000gp/T/ipykernel_3204/2284556048.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)


run q=0.5


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


run q=1.0


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


run q=2.0


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


run q=4.0


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


,q,micro_f1_mean,micro_f1_std,macro_f1_mean,macro_f1_std,n_repeats
0,0.25,0.343758,0.014778,0.341234,0.014694,5
1,0.50,0.330424,0.004377,0.327932,0.003951,5
2,1.00,0.331879,0.017572,0.329680,0.017820,5
3,2.00,0.337212,0.005706,0.334404,0.004660,5
4,4.00,0.327515,0.014694,0.325614,0.014616,5


## Sweep p

In [7]:
p_values = [0.25, 0.5, 1.0, 2.0, 4.0]
p_csv = results_dir / "hmrw_p_sweep.csv"
results_p = run_resumable_sweep(
    varying_param="p",
    values=p_values,
    fixed_params={k: v for k, v in base_params.items() if k != "p"},
    out_csv=p_csv,
)
results_p


run p=0.25


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
/var/folders/8z/x99j21nn30vcxmd912tgx33r0000gp/T/ipykernel_3204/2284556048.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)


run p=0.5


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


run p=1.0


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


run p=2.0


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


run p=4.0


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


,p,micro_f1_mean,micro_f1_std,macro_f1_mean,macro_f1_std,n_repeats
0,0.25,0.332364,0.016215,0.329572,0.016034,5
1,0.50,0.327030,0.016959,0.324920,0.016590,5
2,1.00,0.331879,0.017572,0.329680,0.017820,5
3,2.00,0.331636,0.022646,0.328667,0.022632,5
4,4.00,0.334303,0.009769,0.331448,0.009129,5


## Sweep r

In [8]:
r_values = [0.25, 0.5, 1.0, 2.0, 4.0]
r_csv = results_dir / "hmrw_r_sweep.csv"
results_r = run_resumable_sweep(
    varying_param="r",
    values=r_values,
    fixed_params={k: v for k, v in base_params.items() if k != "r"},
    out_csv=r_csv,
)
results_r


run r=0.25


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
/var/folders/8z/x99j21nn30vcxmd912tgx33r0000gp/T/ipykernel_3204/2284556048.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)


run r=0.5


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


run r=1.0


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


run r=2.0


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


run r=4.0


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


,r,micro_f1_mean,micro_f1_std,macro_f1_mean,macro_f1_std,n_repeats
0,0.25,0.330424,0.007612,0.328207,0.007768,5
1,0.50,0.345455,0.011626,0.343749,0.010818,5
2,1.00,0.331879,0.017572,0.329680,0.017820,5
3,2.00,0.335515,0.011458,0.333833,0.011871,5
4,4.00,0.330182,0.004763,0.328572,0.004674,5


## Sweep num_walks

In [ ]:
num_walks_values = [2, 5, 10, 20, 40]
num_walks_csv = results_dir / "hmrw_num_walks_sweep.csv"
results_num_walks = run_resumable_sweep(
    varying_param="num_walks",
    values=num_walks_values,
    fixed_params={k: v for k, v in base_params.items() if k != "num_walks"},
    out_csv=num_walks_csv,
)
results_num_walks


## Sweep walk_length

In [ ]:
walk_length_values = [10, 20, 40, 80, 120]
walk_length_csv = results_dir / "hmrw_walk_length_sweep.csv"
results_walk_length = run_resumable_sweep(
    varying_param="walk_length",
    values=walk_length_values,
    fixed_params={k: v for k, v in base_params.items() if k != "walk_length"},
    out_csv=walk_length_csv,
)
results_walk_length


## Sweep dim

In [ ]:
dim_values = [16, 32, 64, 128, 256]
dim_csv = results_dir / "hmrw_dim_sweep.csv"
results_dim = run_resumable_sweep(
    varying_param="dim",
    values=dim_values,
    fixed_params={k: v for k, v in base_params.items() if k != "dim"},
    out_csv=dim_csv,
)
results_dim


## Joint Plot (all sweeps)

In [ ]:
def load_and_tag(csv_path, param_name, use_log2=False):
    df = pd.read_csv(csv_path).copy()
    df["param"] = param_name
    if use_log2:
        df["x"] = np.log2(df[param_name].to_numpy())
    else:
        df["x"] = df[param_name].to_numpy()
    return df

joint = pd.concat([
    load_and_tag(results_dir / "hmrw_p_sweep.csv", "p", use_log2=True),
    load_and_tag(results_dir / "hmrw_q_sweep.csv", "q", use_log2=True),
    load_and_tag(results_dir / "hmrw_r_sweep.csv", "r", use_log2=True),
    load_and_tag(results_dir / "hmrw_num_walks_sweep.csv", "num_walks", use_log2=False),
    load_and_tag(results_dir / "hmrw_walk_length_sweep.csv", "walk_length", use_log2=False),
    load_and_tag(results_dir / "hmrw_dim_sweep.csv", "dim", use_log2=False),
], ignore_index=True)

plot_order = ["p", "q", "r", "num_walks", "walk_length", "dim"]
label_map = {
    "p": r"$\log_2(p)$",
    "q": r"$\log_2(q)$",
    "r": r"$\log_2(r)$",
    "num_walks": "num_walks",
    "walk_length": "walk_length",
    "dim": "dim",
}

ymin = min((joint["micro_f1_mean"] - joint["micro_f1_std"]).min(), (joint["macro_f1_mean"] - joint["macro_f1_std"]).min())
ymax = max((joint["micro_f1_mean"] + joint["micro_f1_std"]).max(), (joint["macro_f1_mean"] + joint["macro_f1_std"]).max())

fig, axes = plt.subplots(2, len(plot_order), figsize=(3.6 * len(plot_order), 7), sharey=True)

for j, param in enumerate(plot_order):
    sub = joint[joint["param"] == param].sort_values("x")
    x = sub["x"].to_numpy()

    axes[0, j].plot(x, sub["micro_f1_mean"], marker="o", color="tab:blue")
    axes[0, j].fill_between(x, sub["micro_f1_mean"] - sub["micro_f1_std"], sub["micro_f1_mean"] + sub["micro_f1_std"], alpha=0.2, color="tab:blue")
    axes[0, j].set_title(f"Micro-F1 vs {param}")
    axes[0, j].set_xlabel(label_map[param])

    axes[1, j].plot(x, sub["macro_f1_mean"], marker="o", color="tab:orange")
    axes[1, j].fill_between(x, sub["macro_f1_mean"] - sub["macro_f1_std"], sub["macro_f1_mean"] + sub["macro_f1_std"], alpha=0.2, color="tab:orange")
    axes[1, j].set_title(f"Macro-F1 vs {param}")
    axes[1, j].set_xlabel(label_map[param])

axes[0, 0].set_ylabel("F1")
axes[1, 0].set_ylabel("F1")

for ax in axes.ravel():
    ax.set_ylim(max(0.0, ymin - 0.01), min(1.0, ymax + 0.01))
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_locator(MultipleLocator(0.02))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))

fig.tight_layout()
plt.show()
